# 04b — BiLSTM (and optional TextCNN) training

Trains a 2-layer bidirectional LSTM on the cleaned ViHSD train split using
the PhoW2V-initialised embedding matrix from `04a`. Saves the best checkpoint
by **dev `f1_macro`** (loss is unreliable under the 3-class imbalance), reports
the per-class P/R/F1, and compares against the Week-3 LR + char-ngram champion.

**Decisions made from `04a` Part E that this notebook commits to**:

* `max_len = 64`         — covers >99% of train tokens (p99 ≤ 50, +25% headroom).
* `min_length = 3`       — drops ~11% length-≤2 samples (training only).
* `batch_size = 128`     — sequences are short, larger batch is comfortable.
* `lr = 1e-3`, Adam, `weight_decay = 1e-5`, gradient clip `1.0`.
* `ReduceLROnPlateau` (patience 2) + early stop on `f1_macro` (patience 3).

Outputs:

* `models/dl/bilstm_best.pt`               — full checkpoint (state_dict + config + metrics)
* `models/dl/textcnn_best.pt`              — optional, if RUN_TEXTCNN=True
* `results/bilstm_training_log.json`       — per-epoch metrics
* `results/textcnn_training_log.json`      — optional
* `results/dl_filter_stats.json`           — filter impact
* `results/figures/14_bilstm_training_curves.png`
* `results/figures/15_bilstm_dev_cm.png`
* `results/figures/16_textcnn_training_curves.png` (optional)
* `report/week4_phase1_summary.md`         — auto-generated decision log


In [ ]:
# ── Setup: autoreload, paths, seed ──────────────────────────────────────
%load_ext autoreload
%autoreload 2

# Drop stale src.* caches so edits to src/*.py are picked up without restart.
import sys
for _k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]

import os, json, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path = [str(ROOT)] + [p for p in sys.path if p != str(ROOT)]

from configs.config import (
    PATHS, COLUMNS, LABEL_MAP,
    DEEP_LEARNING_CONFIG, PHOBERT_CONFIG,
)
from src.utils import set_seed, Timer, get_device

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = get_device()

DL_DIR        = ROOT / "models" / "dl"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR   = ROOT / "results"
FIG_DIR       = RESULTS_DIR / "figures"
REPORT_DIR    = ROOT / "report"
for d in (DL_DIR, RESULTS_DIR, FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEXT, LABEL = COLUMNS["text"], COLUMNS["label"]

# 04a Part E decisions ───────────────────────────────────────────────────
MAX_LEN        = 64        # p99 <= 50 in 04a E1; 64 = headroom (multiple of 8)
MIN_LENGTH     = 3         # drop length<=2 samples (~11% of train, 04a E1.2)
BATCH_SIZE     = 128       # sequences short → larger batch fine
EVAL_BATCH     = 256
NUM_WORKERS    = 0         # Jupyter+Windows hang with workers>0; set to 2 elsewhere for ~2x faster data loading
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-5
GRAD_CLIP      = 1.0
MAX_EPOCHS     = 15
ES_PATIENCE    = 3
SCHED_PATIENCE = 2

EMBEDDING_CHOICE      = "phow2v_300d"
EMBEDDING_MATRIX_PATH = DL_DIR / f"embedding_matrix_{EMBEDDING_CHOICE}.pt"

print(f"Device         : {device}")
print(f"Seed           : {RANDOM_STATE}")
print(f"Config         : max_len={MAX_LEN}  min_length={MIN_LENGTH}  batch={BATCH_SIZE}  lr={LEARNING_RATE}")


## 1. Load Week-4 setup artefacts

Re-uses everything `04a` saved: vocab, PhoW2V-aligned embedding matrix,
class weights, cleaned train/dev CSVs. The notebook fails fast if any of
these is missing — re-run `04a` first.

In [ ]:
from src.dataset_dl import Vocab

# vocab ────────────────────────────────────────────────────────────────
VOCAB_PATH = DL_DIR / "vocab.pkl"
assert VOCAB_PATH.exists(), f"Missing {VOCAB_PATH} — run 04a C1 first."
vocab = Vocab.load(VOCAB_PATH)
print(f"vocab           : {len(vocab):,} tokens  (specials {vocab.itos[:4]})")

# embedding matrix ────────────────────────────────────────────────────
assert EMBEDDING_MATRIX_PATH.exists(), f"Missing {EMBEDDING_MATRIX_PATH} — run 04a C3."
_loaded = torch.load(EMBEDDING_MATRIX_PATH, weights_only=True)
emb_matrix = _loaded["embedding"] if isinstance(_loaded, dict) else _loaded
emb_stats  = _loaded.get("stats", {}) if isinstance(_loaded, dict) else {}
EMBEDDING_DIM = int(emb_matrix.shape[1])
print(f"embedding matrix: {tuple(emb_matrix.shape)}  ({EMBEDDING_CHOICE})  coverage={emb_stats.get('coverage', 0)*100:.1f}%")

# class weights ───────────────────────────────────────────────────────
CW_PATH = DL_DIR / "class_weights.pt"
assert CW_PATH.exists(), f"Missing {CW_PATH} — run 04a C4."
class_weights = torch.load(CW_PATH, weights_only=True)
print(f"class weights   : {[f'{LABEL_MAP[i]}={w:.3f}' for i, w in enumerate(class_weights.tolist())]}")

# cleaned splits ──────────────────────────────────────────────────────
train_df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
dev_df   = pd.read_csv(PROCESSED_DIR / "dev_cleaned.csv")
print(f"train_df / dev_df: {len(train_df):,} / {len(dev_df):,} rows")
print(f"  train columns: {list(train_df.columns)}")


## 2. Datasets — filtered train + unfiltered dev

The filter only runs on the training set. Dev stays at the full
distribution so we measure how the model behaves on the real input it
will see at inference time.

In [ ]:
from src.dataset_dl import ViHSDDataset, FilteredViHSDDataset

train_ds_raw = ViHSDDataset(
    texts     = train_df["cleaned"].fillna("").tolist(),
    labels    = train_df[LABEL].tolist(),
    tokenizer = vocab,
    max_len   = MAX_LEN,
    mode      = "bilstm",
)

train_ds = FilteredViHSDDataset(
    base       = train_ds_raw,
    min_length = MIN_LENGTH,
    verbose    = True,
    save_path  = RESULTS_DIR / "dl_filter_stats.json",
)

dev_ds = ViHSDDataset(
    texts     = dev_df["cleaned"].fillna("").tolist(),
    labels    = dev_df[LABEL].tolist(),
    tokenizer = vocab,
    max_len   = MAX_LEN,
    mode      = "bilstm",
)

print(f"\nlen(train_ds_raw) = {len(train_ds_raw):,}")
print(f"len(train_ds)     = {len(train_ds):,}  (after filter)")
print(f"len(dev_ds)       = {len(dev_ds):,}    (no filter — fair eval)")


## 3. DataLoaders

Single deterministic generator per loader so re-running the notebook
reproduces the batch order.

In [ ]:
from src.dataset_dl import collate_fn_bilstm

train_loader = DataLoader(
    train_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    collate_fn  = collate_fn_bilstm,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    generator   = torch.Generator().manual_seed(RANDOM_STATE),
    drop_last   = False,
)

dev_loader = DataLoader(
    dev_ds,
    batch_size  = EVAL_BATCH,
    shuffle     = False,
    collate_fn  = collate_fn_bilstm,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

# Quick sanity peek at one batch.
batch = next(iter(train_loader))
print(f"train batch: input_ids {tuple(batch['input_ids'].shape)} "
      f"lengths {tuple(batch['lengths'].shape)} labels {tuple(batch['labels'].shape)}")
print(f"  lengths range: [{int(batch['lengths'].min())}, {int(batch['lengths'].max())}]")
print(f"  labels: {batch['labels'][:8].tolist()}")


## 4. Model — `BiLSTMClassifier`

Architecture lives in `src/models_dl.py`. Embedding is initialised from
the PhoW2V matrix and left trainable (`freeze_embeddings=False`) so the
~6% missing-vocab rows (random-init) can move toward useful values during
training.

In [ ]:
from src.models_dl import BiLSTMClassifier, count_parameters

set_seed(RANDOM_STATE)
model = BiLSTMClassifier(
    vocab_size            = len(vocab),
    embedding_dim         = EMBEDDING_DIM,
    hidden_dim            = 128,
    num_layers            = 2,
    num_classes           = 3,
    dropout               = 0.3,
    pretrained_embeddings = emb_matrix,
    freeze_embeddings     = False,
    padding_idx           = 0,
).to(device)

param_info = count_parameters(model)


## 5. Loss / optimizer / LR scheduler

Weighted cross-entropy is the main lever for the 3-class imbalance.
`ReduceLROnPlateau` watches dev `f1_macro` (the same metric that early
stopping watches) — when dev F1 plateaus we halve the lr.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode     = "max",
    factor   = 0.5,
    patience = SCHED_PATIENCE,
    min_lr   = 1e-5,
)
print(f"Optimizer : Adam(lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler : ReduceLROnPlateau(factor=0.5, patience={SCHED_PATIENCE}, min_lr=1e-5)")


## 6. Training loop

Helpers first, then the epoch loop. Each epoch logs:

* `train_loss` (sample-weighted average)
* `dev_loss`, `dev_f1_macro`, `dev_f1_clean/off/hate`, `dev_acc`
* current lr, epoch wall time

Best checkpoint = highest `dev_f1_macro`. Early stop if no improvement for
`ES_PATIENCE` epochs.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def run_one_epoch_train(model, loader, criterion, optimizer, grad_clip, device):
    model.train()
    total_loss, total_n = 0.0, 0
    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        lengths   = batch["lengths"]                       # CPU for pack_padded_sequence
        labels    = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, lengths)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n    += bs
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(total_n, 1)


@torch.no_grad()
def run_one_epoch_eval(model, loader, criterion, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc="eval", leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        lengths   = batch["lengths"]
        labels    = batch["labels"].to(device, non_blocking=True)
        logits = model(input_ids, lengths)
        loss   = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_n    += bs
        all_preds.append(logits.argmax(dim=-1).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    metrics = {
        "loss":     total_loss / max(total_n, 1),
        "acc":      float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    per_f = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    metrics["f1_clean"]     = float(per_f[0])
    metrics["f1_offensive"] = float(per_f[1])
    metrics["f1_hate"]      = float(per_f[2])
    return metrics, y_true, y_pred


In [ ]:
BILSTM_CKPT = DL_DIR / "bilstm_best.pt"

torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
best_f1, patience_counter, best_epoch = -1.0, 0, -1
history = []
training_start = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    ep_start = time.perf_counter()
    train_loss = run_one_epoch_train(model, train_loader, criterion, optimizer, GRAD_CLIP, device)
    dev, _, _  = run_one_epoch_eval(model, dev_loader, criterion, device)
    ep_time    = time.perf_counter() - ep_start
    cur_lr     = optimizer.param_groups[0]["lr"]

    row = {
        "epoch":         epoch,
        "train_loss":    float(train_loss),
        "dev_loss":      dev["loss"],
        "dev_acc":       dev["acc"],
        "dev_f1_macro":  dev["f1_macro"],
        "dev_f1_clean":  dev["f1_clean"],
        "dev_f1_off":    dev["f1_offensive"],
        "dev_f1_hate":   dev["f1_hate"],
        "lr":            float(cur_lr),
        "epoch_time_s":  float(ep_time),
    }
    history.append(row)

    scheduler.step(dev["f1_macro"])

    improved = dev["f1_macro"] > best_f1
    if improved:
        best_f1       = dev["f1_macro"]
        best_epoch    = epoch
        patience_counter = 0
        torch.save({
            "epoch":              epoch,
            "model_state_dict":   model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "metrics":            dev,
            "config": {
                "vocab_size":     len(vocab),
                "embedding_dim":  EMBEDDING_DIM,
                "hidden_dim":     128,
                "num_layers":     2,
                "num_classes":    3,
                "dropout":        0.3,
                "max_len":        MAX_LEN,
                "min_length":     MIN_LENGTH,
                "batch_size":     BATCH_SIZE,
                "lr":             LEARNING_RATE,
                "weight_decay":   WEIGHT_DECAY,
                "grad_clip":      GRAD_CLIP,
                "embedding_choice": EMBEDDING_CHOICE,
            },
            "param_info":         param_info,
        }, BILSTM_CKPT)
    else:
        patience_counter += 1

    star = " ★" if improved else ""
    print(f"epoch {epoch:>2d}/{MAX_EPOCHS}  "
          f"train_loss={train_loss:.4f}  "
          f"dev_loss={dev['loss']:.4f}  "
          f"dev_F1m={dev['f1_macro']:.4f}  "
          f"(CL={dev['f1_clean']:.3f} OF={dev['f1_offensive']:.3f} HA={dev['f1_hate']:.3f})  "
          f"lr={cur_lr:.1e}  "
          f"t={ep_time:5.1f}s{star}")

    if patience_counter >= ES_PATIENCE:
        print(f"\n⏹ Early stopping at epoch {epoch} (no improvement for {ES_PATIENCE} epochs).")
        break

total_train_time = time.perf_counter() - training_start
peak_vram = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else 0.0

print(f"\n✓ Training done in {total_train_time:.1f}s")
print(f"  best epoch       : {best_epoch}")
print(f"  best dev f1_macro: {best_f1:.4f}")
print(f"  peak VRAM        : {peak_vram:.2f} GB")
print(f"  checkpoint       : {BILSTM_CKPT}")

# Persist training history.
with open(RESULTS_DIR / "bilstm_training_log.json", "w", encoding="utf-8") as f:
    json.dump({
        "history":          history,
        "best_epoch":       best_epoch,
        "best_f1_macro":    best_f1,
        "total_train_time": total_train_time,
        "peak_vram_gb":     peak_vram,
        "param_info":       param_info,
    }, f, indent=2)
print(f"  log              : {RESULTS_DIR / 'bilstm_training_log.json'}")


## 7. Training curves

Four panels in one figure: train vs dev loss, dev `f1_macro`, per-class
F1, and learning-rate decay.

In [ ]:
import matplotlib.pyplot as plt

hist = pd.DataFrame(history)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

ax = axes[0, 0]
ax.plot(hist["epoch"], hist["train_loss"], "o-", label="train", color="#4C78A8")
ax.plot(hist["epoch"], hist["dev_loss"],   "o-", label="dev",   color="#E45756")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(hist["epoch"], hist["dev_f1_macro"], "o-", color="#54A24B", linewidth=2)
ax.axvline(best_epoch, color="#54A24B", linestyle=":", alpha=0.6, label=f"best @ ep {best_epoch}")
ax.set_xlabel("epoch"); ax.set_ylabel("F1 macro"); ax.set_title("Dev F1 (macro)")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 0]
for col, color, lbl in [
    ("dev_f1_clean", "#4CAF50", "CLEAN"),
    ("dev_f1_off",   "#FF9800", "OFFENSIVE"),
    ("dev_f1_hate",  "#F44336", "HATE"),
]:
    ax.plot(hist["epoch"], hist[col], "o-", color=color, label=lbl)
ax.set_xlabel("epoch"); ax.set_ylabel("F1"); ax.set_title("Per-class dev F1")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.plot(hist["epoch"], hist["lr"], "o-", color="#9D7BBA")
ax.set_xlabel("epoch"); ax.set_ylabel("learning rate"); ax.set_title("LR schedule")
ax.set_yscale("log"); ax.grid(alpha=0.3, which="both")

fig.suptitle(f"BiLSTM training — best dev F1_macro = {best_f1:.4f} @ epoch {best_epoch}",
             fontsize=13, y=1.02)
fig_path = FIG_DIR / "14_bilstm_training_curves.png"
fig.savefig(fig_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"✓ saved → {fig_path}")


## 8. Evaluate the best checkpoint on DEV

Reload state_dict from `bilstm_best.pt` and run a full pass. Reports
classification_report + plots a 3×3 confusion matrix (row-normalised so
diagonal = recall per class).

In [ ]:
from src.evaluate import evaluate_model, plot_confusion_matrix
from sklearn.metrics import classification_report

# Reload best weights (clean separation between training and eval state).
ckpt = torch.load(BILSTM_CKPT, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

t0 = time.perf_counter()
_, y_dev_true, y_dev_pred = run_one_epoch_eval(model, dev_loader, criterion, device)
infer_time = time.perf_counter() - t0

bilstm_metrics = evaluate_model(
    y_true         = y_dev_true,
    y_pred         = y_dev_pred,
    model_name     = "BiLSTM (PhoW2V-300d, 2-layer, hidden=128)",
    train_time     = float(total_train_time),
    inference_time = float(infer_time),
)

print(f"BiLSTM — DEV evaluation (best checkpoint, epoch {ckpt['epoch']})")
print(f"  accuracy : {bilstm_metrics['accuracy']:.4f}")
print(f"  f1_macro : {bilstm_metrics['f1_macro']:.4f}")
print(f"  f1 clean / off / hate : "
      f"{bilstm_metrics['f1_clean']:.4f} / "
      f"{bilstm_metrics['f1_offensive']:.4f} / "
      f"{bilstm_metrics['f1_hate']:.4f}")
print(f"  inference time : {infer_time:.2f}s on {len(y_dev_true):,} dev samples")
print()
print(bilstm_metrics["classification_report"])

cm_path = FIG_DIR / "15_bilstm_dev_cm.png"
plot_confusion_matrix(
    y_true     = y_dev_true,
    y_pred     = y_dev_pred,
    model_name = "BiLSTM (PhoW2V-300d) — DEV",
    save_path  = cm_path,
    normalize  = True,
)
plt.show()
print(f"✓ saved → {cm_path}")


## 9. Compare with the Week-3 LR + char-ngram champion

Numbers for LR are read from `results/week3_final_metrics.json` so they
stay in sync with whatever Week-3 actually produced; fall back to the
quoted values in the spec if the file is unreadable.

In [ ]:
# Week-3 LR champion stats. Try to load from disk first, then fall back.
LR_DEFAULT = {
    "model":         "LR + char-ngram (Week 3)",
    "f1_macro":      0.6367,
    "f1_clean":      0.9208,
    "f1_offensive":  0.4276,
    "f1_hate":       0.5617,
    "train_time":    9.0,
    "param_info":    {"trainable": 25_000, "total": 25_000},
}

WK3_JSON = RESULTS_DIR / "week3_final_metrics.json"
lr_row = LR_DEFAULT.copy()
if WK3_JSON.exists():
    try:
        with open(WK3_JSON, "r", encoding="utf-8") as f:
            wk3 = json.load(f)
        # The shape of this file varies between commits — try common keys.
        candidates = []
        if isinstance(wk3, dict):
            for k, v in wk3.items():
                if isinstance(v, dict) and "f1_macro" in v:
                    candidates.append((k, v))
        # Prefer the entry whose model name mentions char/ngram + LR.
        best = max(candidates, key=lambda kv: (
            ("char" in kv[0].lower() or "char" in str(kv[1].get("model", "")).lower())
            + ("lr" in kv[0].lower() or "logreg" in kv[0].lower()),
            kv[1].get("f1_macro", 0),
        ), default=None)
        if best:
            name, m = best
            lr_row.update({
                "model":        m.get("model", name),
                "f1_macro":     float(m.get("f1_macro", lr_row["f1_macro"])),
                "f1_clean":     float(m.get("f1_clean", lr_row["f1_clean"])),
                "f1_offensive": float(m.get("f1_offensive", lr_row["f1_offensive"])),
                "f1_hate":      float(m.get("f1_hate", lr_row["f1_hate"])),
                "train_time":   float(m.get("train_time", lr_row["train_time"])),
            })
            print(f"Loaded LR champion stats from {WK3_JSON.name} (entry '{name}')")
    except Exception as e:
        print(f"(could not parse {WK3_JSON.name}: {e}; using defaults)")

bilstm_row = {
    "model":         bilstm_metrics["model"],
    "f1_macro":      bilstm_metrics["f1_macro"],
    "f1_clean":      bilstm_metrics["f1_clean"],
    "f1_offensive":  bilstm_metrics["f1_offensive"],
    "f1_hate":       bilstm_metrics["f1_hate"],
    "train_time":    bilstm_metrics["train_time"],
    "param_info":    param_info,
}

comparison_rows = [lr_row, bilstm_row]

def _fmt_row(r):
    p = r.get("param_info", {}) or {}
    return {
        "model":        r["model"],
        "dev_f1_macro": r["f1_macro"],
        "f1_CLEAN":     r["f1_clean"],
        "f1_OFF":       r["f1_offensive"],
        "f1_HATE":      r["f1_hate"],
        "train_s":      r["train_time"],
        "params":       p.get("trainable", r.get("trainable", float("nan"))),
    }

compare_df = pd.DataFrame([_fmt_row(r) for r in comparison_rows])
print("\nComparison vs Week-3 LR champion:\n")
print(compare_df.to_string(index=False,
                            formatters={
                                "dev_f1_macro": "{:.4f}".format,
                                "f1_CLEAN":     "{:.4f}".format,
                                "f1_OFF":       "{:.4f}".format,
                                "f1_HATE":      "{:.4f}".format,
                                "train_s":      "{:.1f}".format,
                                "params":       "{:,.0f}".format,
                            }))

delta_f1_macro = bilstm_row["f1_macro"] - lr_row["f1_macro"]
delta_f1_off   = bilstm_row["f1_offensive"] - lr_row["f1_offensive"]
delta_f1_hate  = bilstm_row["f1_hate"] - lr_row["f1_hate"]
print(f"\nΔ vs LR champion:")
print(f"  f1_macro: {delta_f1_macro:+.4f}  ({delta_f1_macro/max(lr_row['f1_macro'],1e-9)*100:+.1f}%)")
print(f"  f1_OFF  : {delta_f1_off:+.4f}")
print(f"  f1_HATE : {delta_f1_hate:+.4f}")


## 10. (Optional) TextCNN — Phần E

Same data, same hyperparameter budget, different inductive bias.
Set `RUN_TEXTCNN = True` below to actually train; skip otherwise.

In [ ]:
RUN_TEXTCNN = True   # set False to skip Phần E entirely

textcnn_metrics = None
textcnn_param_info = None
textcnn_total_train_time = 0.0
textcnn_best_f1 = -1.0
textcnn_best_epoch = -1

if not RUN_TEXTCNN:
    print("⏭ Skipping TextCNN (RUN_TEXTCNN = False).")
else:
    from src.models_dl import TextCNN

    set_seed(RANDOM_STATE)
    cnn_model = TextCNN(
        vocab_size            = len(vocab),
        embedding_dim         = EMBEDDING_DIM,
        num_filters           = 100,
        kernel_sizes          = (3, 4, 5),
        num_classes           = 3,
        dropout               = 0.5,
        pretrained_embeddings = emb_matrix,
        freeze_embeddings     = False,
        padding_idx           = 0,
    ).to(device)
    textcnn_param_info = count_parameters(cnn_model)

    cnn_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    cnn_optimizer = torch.optim.Adam(cnn_model.parameters(),
                                      lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    cnn_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        cnn_optimizer, mode="max", factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-5,
    )

    CNN_CKPT = DL_DIR / "textcnn_best.pt"
    cnn_history = []
    cnn_patience_counter = 0
    cnn_start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        ep_start = time.perf_counter()
        train_loss = run_one_epoch_train(cnn_model, train_loader, cnn_criterion,
                                          cnn_optimizer, GRAD_CLIP, device)
        dev, _, _  = run_one_epoch_eval(cnn_model, dev_loader, cnn_criterion, device)
        ep_time    = time.perf_counter() - ep_start
        cur_lr     = cnn_optimizer.param_groups[0]["lr"]

        cnn_history.append({
            "epoch": epoch, "train_loss": float(train_loss),
            "dev_loss": dev["loss"], "dev_acc": dev["acc"],
            "dev_f1_macro": dev["f1_macro"],
            "dev_f1_clean": dev["f1_clean"],
            "dev_f1_off":   dev["f1_offensive"],
            "dev_f1_hate":  dev["f1_hate"],
            "lr": float(cur_lr), "epoch_time_s": float(ep_time),
        })

        cnn_scheduler.step(dev["f1_macro"])

        improved = dev["f1_macro"] > textcnn_best_f1
        if improved:
            textcnn_best_f1    = dev["f1_macro"]
            textcnn_best_epoch = epoch
            cnn_patience_counter = 0
            torch.save({
                "epoch":            epoch,
                "model_state_dict": cnn_model.state_dict(),
                "metrics":          dev,
                "config": {
                    "vocab_size":   len(vocab),
                    "embedding_dim": EMBEDDING_DIM,
                    "num_filters":  100,
                    "kernel_sizes": [3, 4, 5],
                    "dropout":      0.5,
                    "max_len":      MAX_LEN,
                    "min_length":   MIN_LENGTH,
                    "batch_size":   BATCH_SIZE,
                    "lr":           LEARNING_RATE,
                    "embedding_choice": EMBEDDING_CHOICE,
                },
                "param_info":       textcnn_param_info,
            }, CNN_CKPT)
        else:
            cnn_patience_counter += 1

        star = " ★" if improved else ""
        print(f"[textcnn] epoch {epoch:>2d}/{MAX_EPOCHS}  "
              f"train_loss={train_loss:.4f}  dev_F1m={dev['f1_macro']:.4f}  "
              f"(CL={dev['f1_clean']:.3f} OF={dev['f1_offensive']:.3f} HA={dev['f1_hate']:.3f})  "
              f"lr={cur_lr:.1e}  t={ep_time:5.1f}s{star}")

        if cnn_patience_counter >= ES_PATIENCE:
            print(f"\n⏹ TextCNN early stopping at epoch {epoch}.")
            break

    textcnn_total_train_time = time.perf_counter() - cnn_start
    print(f"\n✓ TextCNN training done in {textcnn_total_train_time:.1f}s  best epoch={textcnn_best_epoch}  best dev_F1m={textcnn_best_f1:.4f}")

    with open(RESULTS_DIR / "textcnn_training_log.json", "w", encoding="utf-8") as f:
        json.dump({
            "history": cnn_history,
            "best_epoch": textcnn_best_epoch,
            "best_f1_macro": textcnn_best_f1,
            "total_train_time": textcnn_total_train_time,
            "param_info": textcnn_param_info,
        }, f, indent=2)

    # Curves + DEV eval
    cnn_ckpt = torch.load(CNN_CKPT, weights_only=False)
    cnn_model.load_state_dict(cnn_ckpt["model_state_dict"])
    cnn_model.eval()
    t0 = time.perf_counter()
    _, y_dev_true, y_dev_pred_cnn = run_one_epoch_eval(cnn_model, dev_loader, cnn_criterion, device)
    cnn_infer_time = time.perf_counter() - t0

    textcnn_metrics = evaluate_model(
        y_true=y_dev_true, y_pred=y_dev_pred_cnn,
        model_name="TextCNN (PhoW2V-300d, k=[3,4,5], 100 filters)",
        train_time=float(textcnn_total_train_time),
        inference_time=float(cnn_infer_time),
    )

    hist_cnn = pd.DataFrame(cnn_history)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    axes[0,0].plot(hist_cnn["epoch"], hist_cnn["train_loss"], "o-", color="#4C78A8", label="train")
    axes[0,0].plot(hist_cnn["epoch"], hist_cnn["dev_loss"],   "o-", color="#E45756", label="dev")
    axes[0,0].set_title("Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
    axes[0,1].plot(hist_cnn["epoch"], hist_cnn["dev_f1_macro"], "o-", color="#54A24B")
    axes[0,1].axvline(textcnn_best_epoch, color="#54A24B", linestyle=":", alpha=0.6)
    axes[0,1].set_title("Dev F1 macro"); axes[0,1].grid(alpha=0.3)
    for col, color, lbl in [("dev_f1_clean","#4CAF50","CLEAN"),
                             ("dev_f1_off","#FF9800","OFFENSIVE"),
                             ("dev_f1_hate","#F44336","HATE")]:
        axes[1,0].plot(hist_cnn["epoch"], hist_cnn[col], "o-", color=color, label=lbl)
    axes[1,0].set_title("Per-class dev F1"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
    axes[1,1].plot(hist_cnn["epoch"], hist_cnn["lr"], "o-", color="#9D7BBA")
    axes[1,1].set_yscale("log"); axes[1,1].set_title("LR schedule"); axes[1,1].grid(alpha=0.3, which="both")
    fig.suptitle(f"TextCNN training — best dev F1_macro = {textcnn_best_f1:.4f} @ epoch {textcnn_best_epoch}", y=1.02)
    fig_path = FIG_DIR / "16_textcnn_training_curves.png"
    fig.savefig(fig_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✓ saved → {fig_path}")

    print(f"\nTextCNN — DEV evaluation:")
    print(f"  f1_macro : {textcnn_metrics['f1_macro']:.4f}")
    print(f"  f1 clean / off / hate : {textcnn_metrics['f1_clean']:.4f} / {textcnn_metrics['f1_offensive']:.4f} / {textcnn_metrics['f1_hate']:.4f}")

    # Three-way comparison.
    cnn_row = {
        "model":         textcnn_metrics["model"],
        "f1_macro":      textcnn_metrics["f1_macro"],
        "f1_clean":      textcnn_metrics["f1_clean"],
        "f1_offensive":  textcnn_metrics["f1_offensive"],
        "f1_hate":       textcnn_metrics["f1_hate"],
        "train_time":    textcnn_metrics["train_time"],
        "param_info":    textcnn_param_info,
    }
    comparison_rows.append(cnn_row)
    compare_df = pd.DataFrame([_fmt_row(r) for r in comparison_rows])
    print("\nThree-way comparison:\n")
    print(compare_df.to_string(index=False,
                                formatters={
                                    "dev_f1_macro": "{:.4f}".format,
                                    "f1_CLEAN":     "{:.4f}".format,
                                    "f1_OFF":       "{:.4f}".format,
                                    "f1_HATE":      "{:.4f}".format,
                                    "train_s":      "{:.1f}".format,
                                    "params":       "{:,.0f}".format,
                                }))


## 11. Decision log — Phần F

Auto-generates `report/week4_phase1_summary.md` populated with the numbers
from this run (LR vs BiLSTM vs optional TextCNN), filter-impact stats,
hyperparameter notes, and a verdict on which DL model to carry into
Phase 2 (PhoBERT).

In [ ]:
def _row_md(model_name, m, params):
    return (f"| {model_name} "
            f"| {m['f1_macro']:.4f} "
            f"| {m['f1_clean']:.4f} "
            f"| {m['f1_offensive']:.4f} "
            f"| {m['f1_hate']:.4f} "
            f"| {m['train_time']:.0f}s "
            f"| {params:,} |")

# Pick the best DL model by dev f1_macro.
best_dl_name, best_dl_f1, best_dl_params = bilstm_metrics["model"], bilstm_metrics["f1_macro"], param_info["trainable"]
if textcnn_metrics is not None and textcnn_metrics["f1_macro"] > best_dl_f1:
    best_dl_name, best_dl_f1, best_dl_params = textcnn_metrics["model"], textcnn_metrics["f1_macro"], textcnn_param_info["trainable"]

filter_stats = json.load(open(RESULTS_DIR / "dl_filter_stats.json"))

lines = []
lines.append("# Week 4 — Phase 1 summary: BiLSTM (+ optional TextCNN)")
lines.append("")
lines.append(f"_Auto-generated by `notebooks/04b_bilstm_train.ipynb` on best dev `f1_macro` checkpoint._")
lines.append("")
lines.append("## 1. Headline results (DEV)")
lines.append("")
lines.append("| Model | F1 macro | F1 CLEAN | F1 OFF | F1 HATE | Train time | Params |")
lines.append("|---|---|---|---|---|---|---|")
lines.append(_row_md(lr_row["model"], lr_row, lr_row.get("param_info", {}).get("trainable", 25_000)))
lines.append(_row_md(bilstm_metrics["model"], bilstm_metrics, param_info["trainable"]))
if textcnn_metrics is not None:
    lines.append(_row_md(textcnn_metrics["model"], textcnn_metrics, textcnn_param_info["trainable"]))

lines.append("")
lines.append(f"**Δ vs LR champion (BiLSTM):** "
             f"f1_macro {delta_f1_macro:+.4f} ({delta_f1_macro/lr_row['f1_macro']*100:+.1f}%), "
             f"f1_OFF {delta_f1_off:+.4f}, "
             f"f1_HATE {delta_f1_hate:+.4f}.")
lines.append("")

# Per-class winner
clean_best = max(comparison_rows, key=lambda r: r["f1_clean"])["model"]
off_best   = max(comparison_rows, key=lambda r: r["f1_offensive"])["model"]
hate_best  = max(comparison_rows, key=lambda r: r["f1_hate"])["model"]
lines.append("## 2. Per-class analysis")
lines.append("")
lines.append(f"- **CLEAN winner**     : {clean_best}")
lines.append(f"- **OFFENSIVE winner** : {off_best}")
lines.append(f"- **HATE winner**      : {hate_best}")
lines.append("")
if delta_f1_off > 0:
    lines.append(f"- BiLSTM improves OFFENSIVE F1 by {delta_f1_off:+.4f} — sequence context "
                 "lets the model catch implicit insults that a unigram/char-ngram model misses.")
if delta_f1_hate > 0:
    lines.append(f"- BiLSTM improves HATE F1 by {delta_f1_hate:+.4f} — the hardest class, "
                 "where the smallest absolute gain still matters for a 3-class macro.")

lines.append("")
lines.append("## 3. Filter impact (training only)")
lines.append("")
lines.append(f"- Filter rule: drop train samples with `length < {filter_stats['min_length']}` "
             f"(post-truncation token count, includes `<bos>`/`<eos>`).")
lines.append(f"- Removed: {filter_stats['removed']:,} / {filter_stats['original_size']:,} samples "
             f"({filter_stats['pct_removed']:.2f}%).")
lines.append(f"- Effective training set: {filter_stats['filtered_size']:,} samples.")
lines.append(f"- Rationale: 04a E1 showed length≤2 has degenerate hidden-state risk under BiLSTM "
             f"and contributes little signal. Dev/test left UNFILTERED for fair evaluation.")
lines.append("")

# Hyperparameter notes
ckpt_cfg = ckpt["config"]
lines.append("## 4. Final hyperparameters")
lines.append("")
lines.append("| Hyperparameter | Value | Source |")
lines.append("|---|---|---|")
lines.append(f"| max_len             | {ckpt_cfg['max_len']}   | 04a E1: p99 ≤ 50, 64 = headroom |")
lines.append(f"| min_length (train)  | {ckpt_cfg['min_length']}    | 04a E1: drop length≤2 |")
lines.append(f"| batch_size          | {ckpt_cfg['batch_size']}  | sequences short → larger batch fine |")
lines.append(f"| lr / weight_decay   | {ckpt_cfg['lr']} / {ckpt_cfg['weight_decay']} | standard Adam |")
lines.append(f"| grad_clip           | {ckpt_cfg['grad_clip']}   | BiLSTM gradient-explosion guard |")
lines.append(f"| LSTM hidden / layers / dropout | {ckpt_cfg['hidden_dim']} / {ckpt_cfg['num_layers']} / {ckpt_cfg['dropout']} | |")
lines.append(f"| embedding           | {ckpt_cfg['embedding_choice']} ({ckpt_cfg['embedding_dim']}d, trainable) | 04a coverage {emb_stats.get('coverage',0)*100:.1f}% |")
lines.append(f"| early stop / scheduler patience | {ES_PATIENCE} / {SCHED_PATIENCE} | on dev f1_macro |")
lines.append("")

# Verdict & next steps
lines.append("## 5. Verdict")
lines.append("")
if best_dl_f1 > lr_row["f1_macro"]:
    lines.append(f"- ✓ **{best_dl_name}** beats LR champion on dev `f1_macro` "
                 f"({best_dl_f1:.4f} vs {lr_row['f1_macro']:.4f}) — proceed to Phase 2 (PhoBERT).")
else:
    lines.append(f"- ⚠ Best DL model ({best_dl_name}, F1={best_dl_f1:.4f}) does NOT beat LR "
                 f"({lr_row['f1_macro']:.4f}). Investigate before Phase 2: more epochs, "
                 f"unfreeze schedule, layer norm, attention head?")
if textcnn_metrics is not None and bilstm_metrics["f1_macro"] >= textcnn_metrics["f1_macro"]:
    lines.append(f"- BiLSTM ≥ TextCNN ({bilstm_metrics['f1_macro']:.4f} vs {textcnn_metrics['f1_macro']:.4f}) — "
                 "sequence model wins on this dataset.")
elif textcnn_metrics is not None:
    lines.append(f"- TextCNN > BiLSTM ({textcnn_metrics['f1_macro']:.4f} vs {bilstm_metrics['f1_macro']:.4f}) — "
                 "local n-gram features dominate sequence context here.")
lines.append("")
lines.append("## 6. Bridge to PhoBERT (Phase 2)")
lines.append("")
lines.append(f"- Static embeddings cap out around dev_F1_macro≈{best_dl_f1:.3f}. PhoBERT brings "
             "contextual sub-word embeddings + 12-layer transformer attention — historical "
             "results on ViHSD put it at F1_macro ≈ 0.66-0.69 (Luu et al., 2021).")
lines.append("- Memory profile from 04a E2: `bs=16, max_len=128, fp16=True` fits 6 GB with ~2.8 GB peak, "
             "leaving room for the val pass. That's the config 04c will start with.")
lines.append("- Filter rule stays consistent: training-only `min_length=3` "
             "(PhoBERT BPE will rarely produce <3 tokens on raw text, but the conservative cut "
             "keeps comparison fair across phases).")
lines.append("")
lines.append("## 7. Artefacts")
lines.append("")
lines.append(f"- Best checkpoint: `models/dl/bilstm_best.pt` (epoch {ckpt['epoch']}, "
             f"dev F1_macro {best_f1:.4f}, {param_info['trainable']:,} trainable params)")
if textcnn_metrics is not None:
    lines.append(f"- TextCNN checkpoint: `models/dl/textcnn_best.pt` "
                 f"(epoch {textcnn_best_epoch}, dev F1_macro {textcnn_best_f1:.4f})")
lines.append(f"- Training log: `results/bilstm_training_log.json`")
lines.append(f"- Filter stats: `results/dl_filter_stats.json`")
lines.append(f"- Training curves: `results/figures/14_bilstm_training_curves.png`")
lines.append(f"- DEV confusion: `results/figures/15_bilstm_dev_cm.png`")
if textcnn_metrics is not None:
    lines.append(f"- TextCNN curves: `results/figures/16_textcnn_training_curves.png`")

summary_path = REPORT_DIR / "week4_phase1_summary.md"
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"✓ saved → {summary_path}")
print()
print(summary_path.read_text(encoding="utf-8"))
